## Imports

In [1]:
%%capture
%pip install --upgrade google-genai

In [2]:
%%capture
%pip install google-cloud-secret-manager

In [3]:
%%capture
import pandas as pd

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import numpy as np 
from google.cloud import bigquery

import time

from google.cloud.exceptions import NotFound
import json
import os
import re
import io
import math

from datetime import timedelta
from dateutil.relativedelta import relativedelta
from datetime import date, timedelta, datetime

from functools import partial

In [4]:
%%capture
# Google packages
from google.cloud import secretmanager, storage
from google.oauth2 import service_account

# Gemini packages
from google import genai
from google.genai import types
import base64

In [5]:
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 1.26.4
Pandas version: 2.2.2


## Configs

In [6]:
GOOGLE_CLOUD_PROJECT = "pcln-pl-busdatasci-prod"

# gemini model
model = "gemini-2.5-pro"
#model = "gemini-3-pro-preview"
#model = "gemini-2.5-flash"

BATCH_SIZE = 100
OUTPUT_CSV = "hotel_segments_output.csv"
CHECKPOINT_JSON = "hotel_segments_checkpoint.json"

# Model

## Queries

In [7]:
sql_query = '''

SELECT * FROM `pcln-pl-busdatasci-prod.commercial_strategy.commstrat_pcln_hotel_dim_order`
WHERE hotel_id IS NOT NULL
ORDER BY gross_units DESC;

'''

In [8]:
bq_client = bigquery.Client(project=GOOGLE_CLOUD_PROJECT)
df = bq_client.query(sql_query).to_dataframe()

In [46]:
path = "hotel_segments_output.csv"

df_csv = pd.read_csv(path)

# make sure both keys are comparable (numeric is usually safest)
df_ids = pd.to_numeric(df["hotel_id"], errors="coerce")
csv_ids = pd.to_numeric(df_csv["Hotel id"], errors="coerce")  # adjust if your column is named differently

df_remaining = df[~df_ids.isin(csv_ids)].copy()

In [47]:
len(df_remaining)

5

In [48]:
top_df = df_remaining.head(50000).copy()


### Gemini Segmentation Calls

In [49]:
INPUT_COLS = [
    "hotel_id",
    "hotel_name",
    "hotel_latitude",
    "hotel_longitude",
    "brand_owner",
    "hotel_state",
    "hotel_country",
    "chain_scale_segment",
    "hotel_property_type",
    "hotel_address",
    "hotel_zip_code",
]

# -----------------------------
# 1) LLM CALL: refactor generate() to take pipe_table_str
# -----------------------------
def generate(pipe_table_str: str) -> str:

    client = genai.Client(
        vertexai=True,
        project="pcln-pl-busdatasci-prod",
        location="global",
    )

    text1 = types.Part.from_text(text=f"""
        <role>
        You are an expert Hospitality Data Analyst. Your task is to categorize a list of hotel properties into two distinct classification layers: Primary Location and Emerging Trend. Do not hallucinate.
        </role>

        <Layer 1>
        Primary Location (Mandatory - Select One Only)
        Analyze the Address, Zip, Lat/Long, and Brand to assign exactly one of these categories:

        Urban: High-density City Center/CBD. Keywords: "Downtown," "Square," "Plaza."
        Suburban: Outskirts of a major metro, near corporate parks or shopping hubs.
        Airport: Within 5 miles of a major hub. Look for IATA codes (e.g., ORD, LHR) or "Airport" in the name.
        Interstate: Within 1 mile of a major highway. Look for "I-" or "Hwy" in the address.
        Resort: Destination-focused (Beach, Mountain, Theme Park). High-tier Chain Scale + leisure amenities.
        Small Metro/Town: Lower density areas, rural towns, or isolated small cities.
        </Layer 1>

        <Layer 2>
        Emerging Trend Segment (Optional - Apply if Applicable)
        Identify if the property aligns with these 2026 travel trends based on Brand and Name:

        Wellness/Eco: Focused on health, nature, or sustainability (e.g., Six Senses, 1 Hotels, "Retreat").
        Extended Stay: Built for long-term "Bleisure" or Digital Nomads (e.g., Element, Residence Inn, Staybridge).
        Lifestyle/Boutique: High-design, hyper-local, or historic "Character" hotels (e.g., W Hotels, Edition, Curio Collection).
        Work-From-Anywhere: Properties specifically mentioning "Suites" or "Business Hubs" in non-urban settings.
        If no trend applies, mark as "Standard."
        </Layer 2>

        <DATA>
        Data will have the following columns:
        hotel_id|hotel_name|hotel_latitude|hotel_longitude|brand_owner|hotel_state|hotel_country|chain_scale_segment|hotel_property_type|hotel_address|hotel_zip_code
        {pipe_table_str}
        </DATA>

        <Output>
        Please provide a Markdown Table with these columns:
        Hotel id
        Primary Location Segment
        Trend Segment (Use "Standard" if none apply)

        Do not output introductory text before the markdown table.
        Print 'Output Complete' when finished
        </Output>
        """)

    si_text1 = "You are an expert Hospitality Data Analyst."

    contents = [types.Content(role="user", parts=[text1])]

    generate_content_config = types.GenerateContentConfig(
        temperature=0,
        top_p=1,
        max_output_tokens=60192,
        safety_settings=[
            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
        ],
        system_instruction=[types.Part.from_text(text=si_text1)],
    )

    full_output = ""
    for chunk in client.models.generate_content_stream(
        model=model,
        contents=contents,
        config=generate_content_config,
    ):
        if chunk.text:
            full_output += chunk.text

    return full_output

# -----------------------------
# 2) PARSE markdown -> DataFrame (your logic, wrapped)
# -----------------------------
def parse_llm_markdown(email_summary: str) -> pd.DataFrame:
    # Validate marker
    if not re.search(r"(?mi)^\s*Output Complete\s*$", email_summary.strip()):
        raise ValueError("Missing required 'Output Complete' marker. Failing as requested.")

    # Remove marker
    table_text = re.sub(r"(?mi)^\s*Output Complete\s*$", "", email_summary).strip()

    # Parse markdown table
    lines = [ln for ln in table_text.splitlines() if ln.strip()]
    lines = [ln for ln in lines if not re.match(r"^\s*\|\s*:?-{3,}:?\s*\|", ln)]  # drop alignment row
    cleaned = "\n".join(lines)

    df_out = pd.read_csv(
        io.StringIO(cleaned),
        sep=r"\s*\|\s*",
        engine="python"
    )

    # Drop empty cols created by leading/trailing pipes
    df_out = df_out.loc[:, ~df_out.columns.str.match(r"^Unnamed")]

    # Tidy
    df_out.columns = [c.strip() for c in df_out.columns]
    if "Hotel id" in df_out.columns:
        df_out["Hotel id"] = pd.to_numeric(df_out["Hotel id"], errors="coerce").astype("Int64")

    return df_out

# -----------------------------
# 3) Build pipe-table string for a batch
# -----------------------------
def make_pipe_table_str(batch_df: pd.DataFrame) -> str:
    batch_df = batch_df.copy()
    batch_df = batch_df[INPUT_COLS]  # enforce order

    header = "|".join(batch_df.columns)
    rows = batch_df.fillna("").astype(str).agg("|".join, axis=1)
    return header + "\n" + "\n".join(rows.tolist())

# -----------------------------
# 4) Resume logic: skip already processed hotel_ids
# -----------------------------
def get_already_processed_ids(output_csv: str) -> set:
    if not os.path.exists(output_csv):
        return set()
    try:
        existing = pd.read_csv(output_csv)
        # LLM output uses "Hotel id"
        if "Hotel id" in existing.columns:
            return set(pd.to_numeric(existing["Hotel id"], errors="coerce").dropna().astype(int).tolist())
        return set()
    except Exception:
        # If file is corrupted/partial, you can manually fix or delete it.
        return set()

# -----------------------------
# 5) Main loop
# -----------------------------
def run_batches(df: pd.DataFrame):
    # Ensure required cols exist
    missing = [c for c in INPUT_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Input df missing columns: {missing}")

    # Normalize hotel_id for matching
    df2 = df.copy()
    df2["hotel_id"] = pd.to_numeric(df2["hotel_id"], errors="coerce").astype("Int64")
    df2 = df2.dropna(subset=["hotel_id"]).copy()
    df2["hotel_id_int"] = df2["hotel_id"].astype(int)

    processed_ids = get_already_processed_ids(OUTPUT_CSV)
    to_do = df2[~df2["hotel_id_int"].isin(processed_ids)].copy()

    total_rows = len(df2)
    remaining_rows = len(to_do)

    if remaining_rows == 0:
        print(f"Nothing to do. {len(processed_ids)} hotels already in {OUTPUT_CSV}.")
        return

    total_batches = math.ceil(remaining_rows / BATCH_SIZE)
    print(f"Total hotels: {total_rows} | Remaining: {remaining_rows} | Batches to run: {total_batches}")

    # Iterate in batches
    for batch_idx, start in enumerate(range(0, remaining_rows, BATCH_SIZE), start=1):
        batch = to_do.iloc[start:start + BATCH_SIZE].copy()

        # Progress tracker
        done_batches = batch_idx - 1
        print(f"\nBatch {batch_idx}/{total_batches} | "
              f"Hotels in batch: {len(batch)} | "
              f"Completed batches: {done_batches} | Remaining batches: {total_batches - done_batches}")

        # 1) pipe string
        pipe_table_str = make_pipe_table_str(batch)

        # 2) LLM call
        try:
            llm_raw = generate(pipe_table_str)
        except Exception as e:
            print(f"❌ LLM call failed on batch {batch_idx}: {e}")
            # Save checkpoint and stop (so you can rerun)
            with open(CHECKPOINT_JSON, "w") as f:
                json.dump({"last_completed_batch": batch_idx - 1}, f, indent=2)
            raise

        # 3) parse back to df
        try:
            df_out = parse_llm_markdown(llm_raw)
        except Exception as e:
            print(f"❌ Parsing failed on batch {batch_idx}: {e}")
            # Optionally dump the raw output for debugging
            debug_path = f"llm_raw_batch_{batch_idx}.txt"
            with open(debug_path, "w") as f:
                f.write(llm_raw)
            print(f"Saved raw LLM output to {debug_path}")
            with open(CHECKPOINT_JSON, "w") as f:
                json.dump({"last_completed_batch": batch_idx - 1}, f, indent=2)
            raise

        # Optional: add datasource/metadata, or join back to input fields here if desired

        # Append to CSV (safe progress after each batch)
        write_header = not os.path.exists(OUTPUT_CSV)
        df_out.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)

        # Update checkpoint
        with open(CHECKPOINT_JSON, "w") as f:
            json.dump(
                {
                    "last_completed_batch": batch_idx,
                    "batches_total": total_batches,
                    "rows_total": int(total_rows),
                    "rows_remaining_at_start": int(remaining_rows),
                    "output_csv": OUTPUT_CSV,
                },
                f,
                indent=2
            )

        print(f"✅ Saved batch {batch_idx} to {OUTPUT_CSV} (rows written: {len(df_out)})")

        # Optional: light throttling
        time.sleep(0.5)

    print(f"\nAll done. Results appended to: {OUTPUT_CSV}")



In [50]:
# ---- Run it ----
run_batches(top_df)

Total hotels: 5 | Remaining: 5 | Batches to run: 1

Batch 1/1 | Hotels in batch: 5 | Completed batches: 0 | Remaining batches: 1
✅ Saved batch 1 to hotel_segments_output.csv (rows written: 5)

All done. Results appended to: hotel_segments_output.csv


### BQ output

In [51]:
# Set up your Google Cloud project ID
project_id = 'pcln-pl-busdatasci-prod'
dataset_id = 'commercial_strategy'
table_id = 'hotel_industry_trend_segments'
table_full_path = f'{project_id}.{dataset_id}.{table_id}'

# Initialize a client
client = bigquery.Client(project=project_id)

# Ensure the dataset exists (optional, create if not existing)
dataset_ref = bigquery.DatasetReference(project_id, dataset_id)
dataset = bigquery.Dataset(dataset_ref)
try:
    client.get_dataset(dataset_ref)  # Make an API request.
except NotFound:
    print('Check dataset_id')

In [52]:
path = "hotel_segments_output.csv"

df_csv = pd.read_csv(path)


In [53]:
df_csv

,Hotel id,Primary Location Segment,Trend Segment
0,93795103,Resort,Wellness/Eco
1,82003904,Small Metro/Town,Wellness/Eco
2,209004204,Small Metro/Town,Wellness/Eco
3,114218604,Suburban,Standard
4,46309905,Urban,Extended Stay
...,...,...,...
373098,127562604,Resort,Lifestyle/Boutique
373099,204673304,Small Metro/Town,Standard
373100,241751804,Resort,Wellness/Eco
373101,54920306,Resort,Work-From-Anywhere


In [54]:
df_csv = df_csv.rename(columns={
    "Hotel id": "hotel_id",
    "Primary Location Segment": "location_segment",
    "Trend Segment": "trend_segment"
})

In [55]:
import pandas_gbq
pandas_gbq.to_gbq(df_csv, table_full_path, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 15534.46it/s]


## Troubleshooting error batches

In [21]:
i = df.index[df["hotel_id"] == '12907003'][0]   # first match
before = df.loc[:i-1]
after  = df.loc[i+1:]